# 🛣️ RoadSign Evaluator — Modelo de SEÑALIZACIÓN HORIZONTAL

Entrena un modelo separado para **marcas viales** (pintura en el asfalto): pasos de cebra, líneas continuas/discontinuas, doble línea, carril bus, flechas y zigzag.

Este es el **segundo modelo** de tu app, especializado en lo horizontal. La app lo usará junto al de señales verticales.

---

## ⚙️ ANTES DE EMPEZAR

1. **GPU:** Entorno de ejecución → Cambiar tipo → **GPU T4**
2. **API key de Roboflow** (gratis, la misma de antes): pégala en el Paso 2
3. Luego: **Entorno de ejecución → Ejecutar todo**

## ⚠️ Nota honesta sobre la dificultad

Las marcas viales son MÁS difíciles de detectar que las señales verticales (se ven en perspectiva, sobre el suelo, con desgaste). Los datasets disponibles son más pequeños. Espera una precisión algo menor que en verticales — es la naturaleza del problema, no un fallo del entrenamiento. Combinamos varios datasets para maximizar la robustez.

## Paso 1 — GPU + herramientas

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print('✅ GPU activa' if 'GPU' in gpu.stdout or 'T4' in gpu.stdout else '⚠️ Sin GPU: Entorno→Cambiar tipo→GPU T4')
%pip install -q ultralytics roboflow onnx onnxslim pyyaml
print('✅ Herramientas instaladas')

## Paso 2 — Descargar y combinar datasets de marcas viales

Combinamos el dataset MVI (9 clases de marcas, excelente calidad) con datasets de pasos de peatones más grandes para reforzar la clase más común. Pega tu API key abajo.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TU_API_KEY = "TU_API_KEY"
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from roboflow import Roboflow
import os
rf = Roboflow(api_key=TU_API_KEY)

datasets = []

# Dataset principal: Road Markings MVI (9 clases de marcas viales)
try:
    p1 = rf.workspace('mvi-89ske').project('road-markings-o6ofy')
    d1 = p1.version(1).download('yolov8')
    datasets.append(d1.location)
    print(f'✅ Road Markings MVI: {d1.location}')
except Exception as e:
    print(f'⚠️ MVI no disponible: {e}')

# Refuerzo de pasos de peatones (dataset más grande)
try:
    p2 = rf.workspace('yolo-datasets-f9og9').project('crosswalks-zn9wq-kfndo')
    d2 = p2.version(1).download('yolov8')
    datasets.append(d2.location)
    print(f'✅ Crosswalks: {d2.location}')
except Exception as e:
    print(f'⚠️ Crosswalks no disponible: {e}')

print(f'\n{len(datasets)} datasets descargados')
if not datasets:
    raise RuntimeError('No se pudo descargar ningún dataset. Revisa tu API key.')

## Paso 3 — Unificar los datasets en uno solo

Combinamos las imágenes y armonizamos las clases en un único conjunto de entrenamiento. Este paso es delicado: alinea las etiquetas de ambos datasets.

In [ ]:
import os, shutil, yaml, glob

# Conjunto de clases unificado para señalización horizontal española
UNIFIED_CLASSES = [
    'paso_peatones',        # 0 - zebra crossing / paso de cebra
    'linea_continua',       # 1 - solid line (blanca o amarilla)
    'linea_discontinua',    # 2 - broken/dashed line
    'doble_linea',          # 3 - double solid line
    'carril_bus',           # 4 - bus lane marking
    'flecha_carril',        # 5 - turning/direction arrow
    'zigzag',               # 6 - zigzag line
    'linea_detencion',      # 7 - stop line
]

# Mapeo de nombres originales → clase unificada (heurístico por nombre)
def map_class_name(original):
    o = original.lower().replace('-', ' ').replace('_', ' ')
    if 'zebra' in o or 'crosswalk' in o or 'cross walk' in o or 'paso' in o or 'pedestrian cross' in o:
        return 0
    if 'double' in o:
        return 3
    if 'zigzag' in o or 'zig zag' in o:
        return 6
    if 'bus' in o:
        return 4
    if 'turn' in o or 'arrow' in o or 'flecha' in o:
        return 5
    if 'stop line' in o or 'stop-line' in o or 'detencion' in o or 'detention' in o:
        return 7
    if 'broken' in o or 'dash' in o or 'discontinu' in o:
        return 2
    if 'solid' in o or 'continu' in o or 'line' in o:
        return 1
    return None  # clase no reconocida → se ignora

# Carpeta unificada
MERGED = '/content/merged_horizontal'
for split in ['train', 'valid']:
    os.makedirs(f'{MERGED}/{split}/images', exist_ok=True)
    os.makedirs(f'{MERGED}/{split}/labels', exist_ok=True)

counter = 0
for ds_path in datasets:
    # Leer las clases originales de este dataset
    yaml_file = os.path.join(ds_path, 'data.yaml')
    if not os.path.exists(yaml_file):
        continue
    with open(yaml_file) as f:
        ds_cfg = yaml.safe_load(f)
    orig_names = ds_cfg.get('names', [])
    if isinstance(orig_names, dict):
        orig_names = [orig_names[k] for k in sorted(orig_names.keys())]

    # Construir tabla de remapeo de índices
    remap = {}
    for i, name in enumerate(orig_names):
        new_idx = map_class_name(name)
        if new_idx is not None:
            remap[i] = new_idx

    # Copiar imágenes y reescribir labels
    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(ds_path, split, 'images')
        lbl_dir = os.path.join(ds_path, split, 'labels')
        if not os.path.isdir(img_dir):
            continue
        target_split = 'valid' if split in ('valid', 'test') else 'train'
        for img in glob.glob(f'{img_dir}/*'):
            base = os.path.splitext(os.path.basename(img))[0]
            lbl = os.path.join(lbl_dir, base + '.txt')
            # Reescribir el label con clases remapeadas
            new_lines = []
            if os.path.exists(lbl):
                with open(lbl) as lf:
                    for line in lf:
                        parts = line.split()
                        if not parts:
                            continue
                        old_c = int(parts[0])
                        if old_c in remap:
                            parts[0] = str(remap[old_c])
                            new_lines.append(' '.join(parts))
            if new_lines:  # solo guardar si quedan etiquetas válidas
                counter += 1
                ext = os.path.splitext(img)[1]
                shutil.copy(img, f'{MERGED}/{target_split}/images/img{counter}{ext}')
                with open(f'{MERGED}/{target_split}/labels/img{counter}.txt', 'w') as of:
                    of.write('\n'.join(new_lines))

# Crear data.yaml unificado
merged_yaml = {
    'train': f'{MERGED}/train/images',
    'val': f'{MERGED}/valid/images',
    'nc': len(UNIFIED_CLASSES),
    'names': UNIFIED_CLASSES,
}
with open(f'{MERGED}/data.yaml', 'w') as f:
    yaml.dump(merged_yaml, f)

print(f'✅ {counter} imágenes unificadas con {len(UNIFIED_CLASSES)} clases')
print(f'Clases: {UNIFIED_CLASSES}')

# Contar imágenes por split
n_train = len(glob.glob(f'{MERGED}/train/images/*'))
n_val = len(glob.glob(f'{MERGED}/valid/images/*'))
print(f'Train: {n_train} | Val: {n_val}')
if n_train < 50:
    print('⚠️ Muy pocas imágenes. El modelo será limitado. Considera añadir más datasets.')

## Paso 4 — Entrenar

Con datasets pequeños usamos más épocas y augmentation fuerte, partiendo de YOLOv8s pre-entrenado.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
results = model.train(
    data=f'{MERGED}/data.yaml',
    epochs=200,            # más épocas porque hay menos datos
    patience=40,
    imgsz=640,
    batch=16,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.5,   # variación fuerte de luz (asfalto soleado/sombra)
    degrees=8, translate=0.15, scale=0.5,
    perspective=0.0005,     # ligera perspectiva (marcas vistas desde el coche)
    fliplr=0.5,             # las marcas SÍ pueden voltearse horizontalmente
    mosaic=1.0, mixup=0.1,
    cos_lr=True, lr0=0.01,
    project='roadsign_train', name='horizontal', exist_ok=True,
)
print('✅ Entrenamiento completado')

## Paso 5 — Métricas y exportación a ONNX

In [ ]:
import shutil, json, os
best = YOLO('roadsign_train/horizontal/weights/best.pt')
metrics = best.val(data=f'{MERGED}/data.yaml')
print(f'📊 mAP50: {metrics.box.map50:.3f} | mAP50-95: {metrics.box.map:.3f}')

# Exportar
onnx_path = best.export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False)
shutil.move(onnx_path, 'model_horizontal.onnx')
names = best.names
labels = [names[i] for i in range(len(names))]
with open('labels_horizontal.json', 'w', encoding='utf-8') as f:
    json.dump(labels, f, ensure_ascii=False, indent=2)
size_mb = os.path.getsize('model_horizontal.onnx')/1024/1024
print(f'✅ model_horizontal.onnx ({size_mb:.1f} MB) + labels_horizontal.json')
print(f'Clases: {labels}')

## Paso 6 — Descargar

Sube `model_horizontal.onnx` y `labels_horizontal.json` a la carpeta `models/` de tu repo, junto al modelo vertical.

In [ ]:
from google.colab import files
files.download('model_horizontal.onnx')
files.download('labels_horizontal.json')
print('✅ Sube ambos a models/ en tu repo')